In [ ]:
#| default_exp main01

## Chat UI (DaisyUI)

Pure [DaisyUI](https://daisyui.com/) — **no MonsterUI / FrankenUI**.
MonsterUI already loads DaisyUI under the hood, so swapping headers alone looks identical.
This notebook uses a different theme (`nord`), navbar, a Chat/Graph taskbar under the input, a viz-picker pill on the message box, and chat avatars so the difference is obvious.

Messages go to the in-process `string_therapy` `/controller`.

In [ ]:
#| export
import json
import uuid
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path

from dotenv import load_dotenv
from fasthtml.common import *
from starlette.testclient import TestClient
from string_therapy import create_app

REPO = Path(__file__).resolve().parents[1]
if not (REPO / ".env").exists() and (REPO.parent / ".env").exists():
    REPO = REPO.parent
load_dotenv(REPO / ".env")

_controller = TestClient(create_app())

DAISY_THEME = "nord"

daisy_hdrs = (
    Script(src="https://cdn.tailwindcss.com"),
    Link(
        rel="stylesheet",
        href="https://cdn.jsdelivr.net/npm/daisyui@4.12.22/dist/full.min.css",
        type="text/css",
    ),
    Script(src="https://unpkg.com/lucide@0.468.0"),
    Script(src="https://cdn.plot.ly/plotly-2.35.2.min.js"),
    Style("""
      html, body { height: 100%; margin: 0; }
      #main-panel { min-height: 0; }
      .daisy-shell {
        background:
          radial-gradient(ellipse at top, oklch(92% 0.04 230 / 0.9), transparent 55%),
          linear-gradient(180deg, oklch(96% 0.02 240), oklch(98% 0.01 220));
      }
      .chat-bubble { max-width: 36rem; }
      #viz-picker { flex-wrap: wrap; max-width: min(100%, 40rem); justify-content: center; }
      [id$="-chart"] { width: 100%; height: 100%; min-height: 320px; }
    """),
)


In [ ]:
#| export
class IconKind(Enum):
    INTERFACE = "interface"
    IO = "io"

@dataclass
class Icon:
    id: str
    label: str
    icon: str
    kind: IconKind
    active: bool = False
    extra: dict = field(default_factory=dict)

interface_icons = [
    Icon("chat", "Chat", "message-circle", IconKind.INTERFACE, active=True),
    Icon("graph", "Graph", "share-2", IconKind.INTERFACE),
]

GRAPH_HTML = REPO / "ui" / "graph_network.html"
GRAPH_JSON = REPO / "routes" / "router_graph.json"

# Lucide + short labels for endpoint intents (strip category_ prefix → id)
_IO_META = {
    "scatter": ("Scatter", "chart-scatter"),
    "heatmap": ("Heatmap", "grid-3x3"),
    "timeseries": ("Timeseries", "line-chart"),
    "encode_pass": ("Encode", "orbit"),
    "attention_heatmap": ("Attention", "brain"),
    "property_diagnostics": ("Parity", "git-compare"),
    "latent_interpolation": ("Latent", "waypoints"),
    "molecular_structure": ("Molecule", "atom"),
    "roundtrip": ("Roundtrip", "refresh-cw"),
    "start_max": ("Start MAX", "play"),
    "load_weights": ("Weights", "hard-drive"),
    "stop_max": ("Stop MAX", "square"),
}

def _io_id_from_intent(intent: str) -> str:
    intent = (intent or "").strip()
    for prefix in ("solubility_", "settings_"):
        if intent.startswith(prefix):
            return intent[len(prefix) :]
    return intent

def _load_io_icons() -> list[Icon]:
    data = json.loads(GRAPH_JSON.read_text(encoding="utf-8"))
    icons: list[Icon] = []
    for node in data.get("nodes", []):
        if node.get("node_type") != "endpoint":
            continue
        intent = node.get("intent") or ""
        io_id = _io_id_from_intent(intent)
        label, lucide = _IO_META.get(io_id, (io_id.replace("_", " ").title(), "chart-scatter"))
        icons.append(
            Icon(
                io_id,
                label,
                lucide,
                IconKind.IO,
                extra={
                    "url": node.get("url") or "",
                    "method": (node.get("method") or "post").lower(),
                    "intent": intent,
                    "description": node.get("description") or "",
                },
            )
        )
    return icons

io_icons = _load_io_icons()

_VIEW_IDS = ("chat-view", "graph-view", *[f"{i.id}-view" for i in io_icons])
_ACTIVE_OUTLINE_CLASSES = ("ring-2", "ring-primary", "text-primary")

VIZ_CLIENT_JS = REPO / "ui" / "viz_client.js"


def lucide_icon(name: str, cls: str = "w-5 h-5"):
    return I(data_lucide=name, cls=cls)

def _show_view_js(view_id: str) -> str:
    parts = [f"document.getElementById('{v}').classList.add('hidden')" for v in _VIEW_IDS]
    parts.append(f"document.getElementById('{view_id}').classList.remove('hidden')")
    return "; ".join(parts)

def _set_active_icon_js(icon_id: str, *, attr: str = "data-taskbar-icon") -> str:
    remove = "".join(f"b.classList.remove('{c}');" for c in _ACTIVE_OUTLINE_CLASSES)
    add = "".join(f"t.classList.add('{c}');" for c in _ACTIVE_OUTLINE_CLASSES)
    return (
        f"document.querySelectorAll('[{attr}]').forEach(b=>{{"
        + remove
        + "});"
        + f"const t=document.querySelector('[{attr}=\"{icon_id}\"]');"
        + f"if(t){{{add}}}"
    )

def _view_for_icon(icon_id: str) -> str:
    return {"chat": "chat-view", "graph": "graph-view"}.get(icon_id, "chat-view")

def top_navbar():
    return Div(
        Div(
            A(
                lucide_icon("sparkles", cls="w-5 h-5"),
                Span("String Therapy", cls="font-semibold tracking-tight"),
                cls="btn btn-ghost text-xl gap-2",
            ),
            cls="flex-1",
        ),
        Div(
            Span("DaisyUI · nord", cls="badge badge-primary badge-outline"),
            cls="flex-none",
        ),
        cls="navbar bg-base-100/80 backdrop-blur border-b border-base-300 px-4 shadow-sm",
    )

def _taskbar_btn(ic: Icon):
    outline = " ".join(_ACTIVE_OUTLINE_CLASSES) if ic.active else ""
    return Button(
        lucide_icon(ic.icon, cls="w-4 h-4"),
        Span(ic.label, cls="text-[10px] leading-tight"),
        type="button",
        title=ic.label,
        id=f"taskbar-icon-{ic.id}",
        data_taskbar_icon=ic.id,
        cls=f"btn btn-ghost btn-sm h-auto py-1.5 px-3 gap-1 flex-col {outline}".strip(),
        onclick="; ".join([
            _set_active_icon_js(ic.id),
            _show_view_js(_view_for_icon(ic.id)),
            "lucide.createIcons();",
        ]),
    )

def _io_icon_btn(ic: Icon, **kw):
    outline = " ".join(_ACTIVE_OUTLINE_CLASSES) if ic.active else ""
    return Button(
        lucide_icon(ic.icon, cls="w-3.5 h-3.5"),
        title=ic.label,
        id=f"io-icon-{ic.id}",
        data_io_icon=ic.id,
        type="button",
        cls=f"btn btn-ghost btn-circle btn-xs {outline}".strip(),
        **kw,
    )

def render_io_icon(ic: Icon):
    return _io_icon_btn(
        ic,
        onclick=f"window.openIoViz({json.dumps(ic.id)});",
    )

def taskbar():
    return Div(
        *[_taskbar_btn(ic) for ic in interface_icons],
        id="taskbar",
        cls="flex items-center justify-center gap-1 mt-2 px-1 py-1 rounded-xl bg-base-100/90 border border-base-300 shadow-sm",
    )

def viz_picker():
    """Hidden until a category node is clicked in the graph iframe."""
    return Div(
        *[render_io_icon(i) for i in io_icons],
        id="viz-picker",
        cls=(
            "absolute left-1/2 -translate-x-1/2 -top-3 z-10 "
            "inline-flex items-center gap-0.5 px-1.5 py-0.5 "
            "rounded-full border border-base-300 bg-base-100 shadow-sm "
            "hidden"
        ),
    )

def chat_bubble(role: str, content: str):
    is_user = role == "user"
    avatar = Div(
        Div(
            lucide_icon("user" if is_user else "bot", cls="w-4 h-4"),
            cls="w-10 rounded-full bg-primary text-primary-content grid place-items-center"
            if is_user
            else "w-10 rounded-full bg-secondary text-secondary-content grid place-items-center",
        ),
        cls="chat-image avatar",
    )
    return Div(
        avatar,
        Div("You" if is_user else "Router", cls="chat-header opacity-70 text-xs mb-1"),
        Div(
            content,
            cls=f"chat-bubble {'chat-bubble-primary' if is_user else 'chat-bubble-secondary'} whitespace-pre-line shadow",
        ),
        cls=f"chat {'chat-end' if is_user else 'chat-start'} px-2",
    )

def oob_chat_bubble(role: str, content: str):
    return Div(
        chat_bubble(role, content),
        Script("lucide.createIcons();"),
        hx_swap_oob="beforeend:#chat-messages",
    )

def chat_area():
    return Div(
        Div(
            Div(
                Span("Ask the materials router", cls="font-medium"),
                P("Routes over Postgres + Apache AGE", cls="text-sm opacity-60"),
                cls="mb-4 px-2",
            ),
            Div(id="chat-messages", cls="flex flex-col gap-3 w-full"),
            cls="w-full max-w-2xl mx-auto py-6 px-3",
        ),
        id="chat-view",
        cls="flex flex-col flex-1 min-h-0 overflow-y-auto",
    )

def graph_view():
    return Div(
        Div(
            Span("Router graph", cls="font-medium"),
            Span("live", cls="badge badge-accent badge-sm ml-2"),
            cls="px-4 py-2 border-b border-base-300 bg-base-100/70",
        ),
        Iframe(src="/graph", title="Router graph", cls="w-full flex-1 border-0 bg-base-100"),
        id="graph-view",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden hidden",
    )

def _viz_view(ic: Icon):
    return Div(
        Div(
            Span(ic.label, cls="font-medium"),
            Span(ic.extra.get("intent") or ic.id, cls="badge badge-ghost badge-sm ml-2"),
            cls="px-4 py-2 border-b border-base-300 bg-base-100/70 flex items-center",
        ),
        Div(
            P(
                ic.extra.get("description") or f"{ic.label} visualization",
                id=f"{ic.id}-status",
                cls="text-sm opacity-60 px-3 pt-3",
            ),
            Div(id=f"{ic.id}-chart", cls="w-full flex-1 min-h-0"),
            cls="flex flex-col flex-1 min-h-0",
        ),
        id=f"{ic.id}-view",
        data_viz_url=ic.extra.get("url") or "",
        data_viz_method=ic.extra.get("method") or "post",
        data_viz_intent=ic.extra.get("intent") or ic.id,
        cls="flex flex-col flex-1 min-h-0 overflow-hidden hidden",
    )

def main_panel():
    return Div(
        chat_area(),
        graph_view(),
        *[_viz_view(i) for i in io_icons],
        id="main-panel",
        cls="flex flex-col flex-1 min-h-0 overflow-hidden pb-40",
    )

def input_bar():
    return Div(
        Form(
            Div(
                viz_picker(),
                Textarea(
                    placeholder="Message the router…",
                    name="message",
                    rows=2,
                    cls="textarea textarea-bordered textarea-primary w-full resize-none rounded-2xl pt-6 pr-14 leading-snug",
                    id="chat-input",
                    onkeydown="if(event.key==='Enter'&&!event.shiftKey){event.preventDefault();this.form.requestSubmit()}",
                    hx_on_input="""
                        let btn = document.getElementById('submit-btn');
                        if (this.value.trim().length > 0) btn.classList.remove('btn-disabled');
                        else btn.classList.add('btn-disabled');
                        """,
                ),
                Button(
                    lucide_icon("send", cls="w-4 h-4"),
                    id="submit-btn",
                    cls="btn btn-primary btn-circle btn-sm absolute right-3 bottom-3 btn-disabled",
                    type="submit",
                ),
                cls="relative w-full",
            ),
            hx_post="/chat/send",
            hx_target="#htmx-sink",
            hx_swap="none",
            hx_on__after_request=(
                "document.getElementById('chat-input').value='';"
                "document.getElementById('submit-btn').classList.add('btn-disabled');"
                "lucide.createIcons();"
            ),
            cls="w-full",
        ),
        taskbar(),
        id="input-bar",
        cls="fixed bottom-0 left-0 right-0 z-10 px-3 pb-3 pt-5 max-w-2xl mx-auto w-full",
    )

app, rt = fast_app(
    hdrs=daisy_hdrs,
    pico=False,
    htmlkw={"data-theme": DAISY_THEME},
    bodykw={"class": "daisy-shell min-h-screen"},
)

@rt("/ui/viz_client.js")
def viz_client_js():
    return FileResponse(VIZ_CLIENT_JS, media_type="application/javascript")

@rt("/graph")
def graph():
    html = GRAPH_HTML.read_text(encoding="utf-8")
    data = json.loads(GRAPH_JSON.read_text(encoding="utf-8"))
    marker = '<script id="graph-data" type="application/json">null</script>'
    payload = f'<script id="graph-data" type="application/json">{json.dumps(data)}</script>'
    if marker not in html:
        raise RuntimeError("graph_network.html missing #graph-data injection point")
    html = html.replace(marker, payload, 1)

    return Response(html, media_type="text/html")

@rt("/")
def get(session):
    if "session_id" not in session:
        session["session_id"] = str(uuid.uuid4())

    return Div(
        top_navbar(),
        main_panel(),
        input_bar(),
        Div(id="htmx-sink", cls="hidden", aria_hidden="true"),
        Script(src="/ui/viz_client.js"),
        Script("lucide.createIcons();"),
        cls="flex flex-col h-screen",
    )

def _icon_id_from_parsed(parsed: dict) -> str | None:
    ep = str(parsed.get("endpoint") or "").strip()
    for prefix in ("solubility_", "settings_"):
        if ep.startswith(prefix):
            return ep[len(prefix) :]
    viz = str(parsed.get("visualization") or "").strip()
    if viz == "parity":
        return "property_diagnostics"
    if viz == "settings_panel":
        action = ((parsed.get("panel") or {}) if isinstance(parsed.get("panel"), dict) else {}).get("action")
        return str(action) if action else "start_max"
    return viz or None


def _chat_reply_for_parsed(parsed: dict | None, raw_reply: str) -> str:
    if isinstance(parsed, dict):
        if parsed.get("status") == "error":
            return f"Visualization failed: {parsed.get('error') or 'unknown error'}"
        if parsed.get("panel"):
            icon = _icon_id_from_parsed(parsed) or "settings"
            note = ""
            panel = parsed.get("panel")
            if isinstance(panel, dict) and panel.get("note"):
                note = f" — {panel['note']}"
            ready = "ready" if (isinstance(panel, dict) and panel.get("ready")) else "not ready"
            return f"Settings: {icon.replace('_', ' ')} ({ready}){note}."
        if parsed.get("plotly") or parsed.get("plot"):
            icon = _icon_id_from_parsed(parsed) or "chart"
            src = parsed.get("source") or ("matgram" if not parsed.get("placeholder") else "demo")
            return f"Opened {icon.replace('_', ' ')} visualization ({src})."
    reply = raw_reply if isinstance(raw_reply, str) else json.dumps(raw_reply, indent=2)
    if reply.strip().lower() == "done":
        return "Done."
    if len(reply) > 500 and (reply.lstrip().startswith("{") or reply.lstrip().startswith("[")):
        return "Request completed."
    return reply


@rt("/chat/send", methods=["POST"])
async def chat_send(message: str, session):
    user_msg = message.strip()
    if not user_msg:
        return ""

    sid = session.get("session_id", "").strip()
    payload: dict = {"message": user_msg, "use_history": True, "response_format": "json"}
    if sid:
        payload["session_id"] = sid

    r = _controller.post("/controller", json=payload)
    try:
        data = r.json()
    except Exception:  # noqa: BLE001
        data = {"detail": r.text[:2000]}

    parsed = data.get("parsed") if isinstance(data, dict) else None
    if r.status_code >= 400:
        reply = data.get("detail") or data.get("error") or f"Controller error HTTP {r.status_code}"
        parsed = None
    else:
        reply = _chat_reply_for_parsed(
            parsed if isinstance(parsed, dict) else None,
            data.get("response") or data.get("detail") or "No response",
        )

    new_sid = (data.get("session_id") or sid or "").strip()
    if new_sid:
        session["session_id"] = new_sid

    parts: list = [
        oob_chat_bubble("user", user_msg),
        oob_chat_bubble("assistant", reply),
        Script(
            "window.__stLastChatMessage = "
            + json.dumps(user_msg)
            + ";"
            "requestAnimationFrame(()=>{const c=document.getElementById('chat-view');"
            "if(c)c.scrollTop=c.scrollHeight; lucide.createIcons();})"
        ),
    ]

    if isinstance(parsed, dict) and (
        parsed.get("plotly") or parsed.get("plot") or parsed.get("panel")
    ):
        icon_id = _icon_id_from_parsed(parsed)
        if icon_id:
            parts.append(
                Script(
                    "window.openIoVizFromChat("
                    + json.dumps(icon_id)
                    + ", "
                    + json.dumps(parsed)
                    + ");"
                )
            )

    return tuple(parts)

if __name__ == "__main__":
    serve(appname="string_therapy_for_material_science.main01", port=4546)


## One-time DB setup (optional)

Run when you need to (re)bootstrap schema + seed the router graph. Not exported into `main01`.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from string_therapy import ensure_schema, seed_router_db

repo = Path.cwd() if (Path.cwd() / '.env').exists() else Path.cwd().parent
load_dotenv(repo / '.env')

ensure_schema()
seed_router_db(repo / 'routes' / 'router_graph.json')
print('schema + seed ok')